## 1. Portfolio KPI Monitoring (MoM / Risk Trends)
This notebook simulates a lightweight retail-credit risk monitoring pack. A month-on-month (MoM) KPI analysis in credit decision analytics.

**Key ideas:**
- Generate 12 months of synthetic credit portfolio data (similar to a cards / personal-loan book)
- Calculate core risk KPIs (Bad rate 30+ DPD, early arrears 1+, approval rate, utilisation, vintage bad)
- Track MoM movements and highlight potential early warning signals
- Visualise key relationships: Early arrears vs Bad rate, Approval rate vs Bad rate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9,4)
plt.rcParams['axes.grid'] = True

## 2. Generate synthetic monthly portfolio data

We simulate a simplified cards / personal-loan portfolio with:
- Applications / approvals / declines
- Balances & limits
- 1+ DPD (early arrears) and 30+ DPD (bad bucket)
- New 30+ entries for a basic vintage view

In [ ]:
# Synthetic raw data (12 months)
np.random.seed(42)

months = pd.date_range('2023-01-01', periods=12, freq='MS')
month_labels = months.strftime('%m-%Y')

# Applications between 2600 - 3400
applications = np.random.randint(2600, 3400, size=12)

# Approval rates between 58% and 78%, with a dip mid-year
base_app_rate = np.array([0.74,0.72,0.70,0.69,0.67,0.65,0.62,0.60,0.63,0.66,0.62,0.61])
noise = np.random.normal(0, 0.01, size=12)
approval_rate = np.clip(base_app_rate + noise, 0.55, 0.80)
approvals = (applications * approval_rate).astype(int)
declines = applications - approvals

# Balances and limits
balances = np.random.randint(11_000_000, 14_000_000, size=12)
limit_multipliers = np.random.uniform(1.7, 2.1, size=12)
limits = (balances * limit_multipliers).astype(int)

# Early arrears (1+ DPD) approx 6-8% of approvals, with some noise
early_pct = np.array([0.065,0.062,0.060,0.058,0.061,0.067,0.074,0.050,0.060,0.055,0.059,0.071])
dpd1 = (approvals * early_pct).astype(int)

# Bad bucket (30+ DPD) ~2.0-2.9% of accounts, correlated with early arrears
bad_pct = np.array([0.022,0.026,0.019,0.021,0.016,0.020,0.026,0.028,0.019,0.022,0.019,0.027])
dpd30 = (approvals * bad_pct).astype(int)

# New 30+ each month as a fraction of early arrears
new30 = (dpd1 * np.array([0.12,0.11,0.13,0.10,0.09,0.11,0.14,0.13,0.10,0.11,0.12,0.14])).astype(int)

raw = pd.DataFrame({'Month': month_labels,'Applications': applications,'Approvals': approvals,'Declines': declines,'Balances': balances,'Limits': limits,'DPD1': dpd1,'DPD30': dpd30, 'New30+': new30})
raw